# G6a — Is the listener (B) observable before B speaks?

Gate for the *listener-aware* formulation. Hi-EF forecasts B's emotion in clip IV from clips I–III, but models only
the speaker. This notebook measures, on **train + val only** (test untouched), how often B can actually be seen:

* **listening:** B's face appears in clip III while A is talking (same frame or a cut-away reaction shot);
* **previous turn:** B spoke in clip II or clip I.

Faces are unreliable for "who spoke" (speakers often turn away, profile faces break identity embeddings) and scenes
may contain a third person, so the notebook uses **two identity channels**:

* **voice** (ECAPA speaker embeddings of each clip's audio) for turn identity: is the voice of clip II / I the same as
  clip IV's (B's) voice? It also measures how often clip III and IV have the *same* voice, i.e. MCIS that violate the
  dataset rule A ≠ B;
* **faces** for listening: B's face in clip III, counted as *usable* only when roughly frontal (|yaw| ≤ MAX_YAW).

Face method, per MCIS: sample frames from clips I–IV, detect faces and compute ArcFace embeddings (InsightFace),
cluster all faces of the MCIS into persons, take the dominant person of each clip as its speaker proxy.
A = dominant person of III, B_true = dominant person of IV (**analysis only**). It also scores an inference-time
rule that picks B **without** clip IV, and writes annotated frame montages for visual checking.

Gate: B observable (usable listening face or previous-turn voice) in ≥ 40–50% of MCIS, and the no-clip-IV rules
(face and voice) are right in ≥ 80% of the cases where they fire. Results are split into two-person and multi-person scenes.

In [ ]:
# insightface can pull the CPU build of onnxruntime, which overwrites onnxruntime-gpu (same package dir).
# Install it first, then remove every onnxruntime build, then install the GPU build last.
!pip install -q insightface speechbrain
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu
!pip list 2>/dev/null | grep -i onnxruntime

In [ ]:
# ======== CONFIG ========
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
SPLIT_CSV = "/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv"
OUT_DIR = "/kaggle/working"

SPLITS = ["train", "val"]    # test stays untouched
N_MCIS = 600                 # random MCIS to analyse (None = all train+val; ~4x slower)
SAMPLE_FPS, MAX_FRAMES = 3, 24
DET_SIZE = (640, 640)
MIN_DET_SCORE, MIN_FACE_PX = 0.6, 24
SAME_PERSON_COS = 0.45       # cosine similarity above which two faces are the same person
DOMINANT_MIN_FRAC = 0.25     # a clip's dominant person must be in >= this fraction of its sampled frames
N_MONTAGES = 24
MAX_YAW = 45                 # degrees; listener faces beyond this are counted as not usable for expression
VOICE_SAME_COS = 0.35        # ECAPA cosine above which two clips are taken as the same speaker
AUDIO_SR = 16000
SEED = 0

In [ ]:
import os, glob, random, json
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from sklearn.cluster import AgglomerativeClustering

roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
AUDIO_ROOTS = [os.path.join(r, "audio") for r in roots if os.path.isdir(os.path.join(r, "audio"))]
ANNOT_CSV = [os.path.join(r, "annotation.csv") for r in roots if os.path.exists(os.path.join(r, "annotation.csv"))][0]
print("video roots:", VIDEO_ROOTS, "| audio roots:", AUDIO_ROOTS)


def audio_path(clip):
    ep, num = clip.split('/')
    for root in AUDIO_ROOTS:
        for ext in ('.mp3', '.wav', '.flac', '.m4a'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


def video_path(clip):
    ep, num = clip.split('/')
    for root in VIDEO_ROOTS:
        for ext in ('.mp4', '.avi', '.mkv', '.mov'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


ann = pd.read_csv(ANNOT_CSV, header=None, dtype=str).set_index(0)
sp = pd.read_csv(SPLIT_CSV, dtype=str)
sp = sp[sp.split.isin(SPLITS)].reset_index(drop=True)
assert 'test' not in set(sp.split)
if N_MCIS is not None and N_MCIS < len(sp):
    sp = sp.sample(n=N_MCIS, random_state=SEED).reset_index(drop=True)
clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()))
missing = [c for c in clips if video_path(c) is None]
print(f"MCIS {len(sp)} | unique clips {len(clips)} | clips without video {len(missing)}", missing[:5])

In [ ]:
import torch  # loads the CUDA/cuDNN libraries that onnxruntime-gpu can reuse
import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception as e:
        print("preload_dlls:", e)
ON_GPU = 'CUDAExecutionProvider' in ort.get_available_providers() and torch.cuda.is_available()
print("onnxruntime providers:", ort.get_available_providers())
if not ON_GPU:
    # CPU fallback: keep the run to a manageable size instead of silently running for hours
    print("WARNING: no CUDA provider for onnxruntime -> CPU mode: fewer MCIS/frames, smaller detector input")
    N_CPU_MCIS = 150
    if len(sp) > N_CPU_MCIS:
        sp = sp.sample(n=N_CPU_MCIS, random_state=SEED).reset_index(drop=True)
        clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()))
    MAX_FRAMES, DET_SIZE = 12, (480, 480)
    print(f"CPU mode: {len(sp)} MCIS, {len(clips)} clips, <= {MAX_FRAMES} frames/clip")

from insightface.app import FaceAnalysis
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition', 'landmark_3d_68'],
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0 if ON_GPU else -1, det_size=DET_SIZE)
print("providers actually used:", {k: m.session.get_providers() for k, m in app.models.items()})


def read_frames(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    want = max(1, min(MAX_FRAMES, int(round(n / fps * SAMPLE_FPS)))) if n else MAX_FRAMES
    frames = []
    for i in sorted(set(np.linspace(0, max(n - 1, 0), want).astype(int).tolist())):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)   # seek instead of decoding the whole clip
        ok, fr = cap.read()
        if ok:
            frames.append(fr)
    cap.release()
    return frames


FACES = {}      # clip -> list of dicts(frame, bbox, score, emb)
NFRAMES = {}
KEEP_FRAMES = {}  # a few frames per clip for montages
montage_mcis = set(sp.sample(n=min(N_MONTAGES, len(sp)), random_state=SEED + 1).sample_id)
montage_clips = set(sp[sp.sample_id.isin(montage_mcis)][['clip1', 'clip2', 'clip3', 'clip4']].values.ravel())
import time as _time
_t0 = _time.time()
for ci, c in enumerate(clips):
    if ci % 100 == 0:
        print(f"faces: {ci}/{len(clips)} clips, {(_time.time() - _t0) / 60:.1f} min", flush=True)
    p = video_path(c)
    frames = read_frames(p) if p else []
    NFRAMES[c] = len(frames)
    out = []
    for fi, fr in enumerate(frames):
        for f in app.get(fr):
            x1, y1, x2, y2 = f.bbox
            if f.det_score >= MIN_DET_SCORE and min(x2 - x1, y2 - y1) >= MIN_FACE_PX:
                pose = getattr(f, 'pose', None)
                out.append({'frame': fi, 'bbox': f.bbox.astype(int).tolist(), 'score': float(f.det_score),
                            'emb': f.normed_embedding.astype(np.float32),
                            'yaw': float(pose[1]) if pose is not None else 0.0})
    FACES[c] = out
    if c in montage_clips and frames:
        pick = np.linspace(0, len(frames) - 1, min(6, len(frames))).astype(int)
        KEEP_FRAMES[c] = [(int(i), frames[i]) for i in pick]
print("clips with >=1 face:", sum(bool(v) for v in FACES.values()), "/", len(FACES))

In [ ]:
def analyse(row):
    cl = [row['clip1'], row['clip2'], row['clip3'], row['clip4']]
    faces = [(k, f) for k, c in enumerate(cl) for f in FACES.get(c, [])]
    res = {'sample_id': row['sample_id'], 'split': row['split'], 'source_folder': row['source_folder'],
           'emo_A': row['clip3_emotion'], 'emo_B': row['clip4_emotion'],
           'unc_B': str(ann.at[row['clip4'], 8]) if row['clip4'] in ann.index else 'NA'}
    if len(faces) == 0:
        return res | {'persons': 0}, {}
    E = np.stack([f['emb'] for _, f in faces])
    lab_ = (np.zeros(1, int) if len(E) == 1 else
            AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                    distance_threshold=1 - SAME_PERSON_COS).fit_predict(E))
    # frames per (clip, person)
    pres = {}
    for (k, f), p in zip(faces, lab_):
        pres.setdefault((k, int(p)), set()).add(f['frame'])

    def dominant(k):
        n = NFRAMES.get(cl[k], 0)
        cand = [(len(fr), p) for (kk, p), fr in pres.items() if kk == k]
        if not cand or n == 0:
            return None
        cnt, p = max(cand)
        return p if cnt / n >= DOMINANT_MIN_FRAC else None

    dom = [dominant(k) for k in range(4)]
    A, Bt = dom[2], dom[3]
    in3 = {p for (k, p) in pres if k == 2}
    others3 = sorted(in3 - {A}, key=lambda p: -len(pres[(2, p)]))
    # inference-time rule (no clip IV): a non-A person of clip III, preferring one who spoke in II or I
    spoke = [p for p in (dom[1], dom[0]) if p is not None and p != A]
    if others3:
        pref = [p for p in others3 if p in spoke]
        Bp = pref[0] if pref else others3[0]
        src = 'III_listening'
    elif spoke:
        Bp, src = spoke[0], 'previous_turn'
    else:
        Bp, src = None, 'none'
    n3 = max(NFRAMES.get(cl[2], 0), 1)
    frontal_B3 = {f['frame'] for (k, f), p in zip(faces, lab_) if k == 2 and Bt is not None and Bt != A and p == Bt
                  and abs(f['yaw']) <= MAX_YAW}
    ids3 = [frozenset(p for (k, p), fr in pres.items() if k == 2 and fi in fr) for fi in range(n3)]
    res |= {
        'persons': len(set(int(p) for p in lab_)), 'persons_III': len(in3), 'A_found': A is not None,
        'Btrue_found': Bt is not None, 'A_eq_Btrue': A is not None and A == Bt,
        'B_listening_III': Bt is not None and Bt != A and Bt in in3,
        'B_frac_III': len(pres.get((2, Bt), ())) / n3 if Bt is not None and Bt != A else 0.0,
        'B_same_frame_as_A': Bt is not None and A is not None and Bt != A and
                             bool(pres.get((2, Bt), set()) & pres.get((2, A), set())),
        'B_listening_III_frontal': len(frontal_B3) > 0,
        'B_spoke_II': Bt is not None and Bt != A and dom[1] == Bt,
        'B_spoke_I': Bt is not None and Bt != A and dom[0] == Bt,
        'Bpred_source': src, 'Bpred_correct': Bp is not None and Bt is not None and Bt != A and Bp == Bt,
        'Bpred_made': Bp is not None,
        'cuts_III': sum(ids3[i] != ids3[i - 1] for i in range(1, len(ids3))),
    }
    res['B_observable'] = res['B_listening_III'] or res['B_spoke_II'] or res['B_spoke_I']
    return res, {'faces': faces, 'labels': lab_, 'A': A, 'Bt': Bt, 'Bp': Bp}


rows, DETAIL = [], {}
for r in tqdm(sp.to_dict('records'), desc='MCIS'):
    res, det = analyse(r)
    rows.append(res)
    if r['sample_id'] in montage_mcis:
        DETAIL[r['sample_id']] = (r, det)
df = pd.DataFrame(rows)
BOOL = ['A_found', 'Btrue_found', 'A_eq_Btrue', 'B_listening_III', 'B_listening_III_frontal', 'B_same_frame_as_A',
        'B_spoke_II', 'B_spoke_I',
        'Bpred_correct', 'Bpred_made', 'B_observable']
for c in BOOL:
    df[c] = df[c].fillna(False).astype(bool) if c in df else False
for c in ['persons_III', 'B_frac_III', 'cuts_III']:
    df[c] = df[c].fillna(0) if c in df else 0
df['Bpred_source'] = df['Bpred_source'].fillna('none') if 'Bpred_source' in df else 'none'
df.to_csv(f"{OUT_DIR}/g6a_listener_visibility.csv", index=False)

In [ ]:
ok = df[df.A_found & df.Btrue_found & ~df.A_eq_Btrue]
print(f"MCIS analysed: {len(df)}")
print(f"  faces found at all: {(df.persons > 0).mean() * 100:.1f}%   A (clip III dominant) found: {df.A_found.mean() * 100:.1f}%   "
      f"B_true (clip IV dominant) found: {df.Btrue_found.mean() * 100:.1f}%")
print(f"  sanity: dominant(III) == dominant(IV) in {df.A_eq_Btrue.mean() * 100:.1f}% (dataset rule says A != B; high = proxy fails)")
print(f"\nAmong {len(ok)} MCIS where A and B_true are identified and distinct:")
for col, name in [('B_listening_III', 'B visible in clip III (listening/reaction)'),
                  ('B_listening_III_frontal', '  ... with a usable (near-frontal) face'),
                  ('B_same_frame_as_A', '  ... in the same frame as A'),
                  ('B_spoke_II', 'B was dominant in clip II'), ('B_spoke_I', 'B was dominant in clip I'),
                  ('B_observable', 'B observable (III listening or I/II turn)')]:
    print(f"  {name:<46} {ok[col].mean() * 100:5.1f}%")
print(f"  mean fraction of clip-III frames showing B: {ok.B_frac_III.mean():.2f}  | mean identity changes in III: {ok.cuts_III.mean():.1f}")
print(f"\nB_observable over ALL analysed MCIS (unidentified counted as not observable): "
      f"{df.B_observable.mean() * 100:.1f}%")
made = ok[ok.Bpred_made]
print(f"\nNo-clip-IV rule: proposes someone in {ok.Bpred_made.mean() * 100:.1f}% of identified MCIS; "
      f"precision {made.Bpred_correct.mean() * 100:.1f}%")
print(made.groupby('Bpred_source').Bpred_correct.agg(['size', 'mean']).rename(columns={'mean': 'precision'}).round(3).to_string())
print("\nB visible while listening, by split / by B emotion:")
print(ok.groupby('split').B_listening_III.mean().round(3).to_string())
print(ok.groupby('emo_B').B_listening_III.agg(['size', 'mean']).round(3).to_string())

## Voice channel: who spoke in clips I–IV (robust to profile faces)

In [ ]:
import librosa, torch
try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier
spk = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=f"{OUT_DIR}/ecapa",
                                     run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"})
VOICE = {}
for c in tqdm(clips, desc='voice'):
    p = audio_path(c)
    if p is None:
        continue
    try:
        wav, _ = librosa.load(p, sr=AUDIO_SR, mono=True)
    except Exception:
        continue
    if len(wav) < AUDIO_SR // 2:
        continue
    with torch.no_grad():
        e = spk.encode_batch(torch.tensor(wav, dtype=torch.float32).unsqueeze(0)).reshape(-1).cpu().numpy()
    VOICE[c] = e / (np.linalg.norm(e) + 1e-9)
print(f"voice embeddings: {len(VOICE)}/{len(clips)} clips", flush=True)


def vcos(a, b):
    return float(VOICE[a] @ VOICE[b]) if a in VOICE and b in VOICE else np.nan


cl_of = sp.set_index('sample_id')[['clip1', 'clip2', 'clip3', 'clip4']]
for x, y in [(3, 4), (2, 4), (1, 4), (2, 3), (1, 3)]:
    df[f'vcos_{x}{y}'] = [vcos(cl_of.at[s_, f'clip{x}'], cl_of.at[s_, f'clip{y}']) for s_ in df.sample_id]
df['multi_party'] = df.persons >= 3
df.to_csv(f"{OUT_DIR}/g6a_listener_visibility.csv", index=False)
print("cosine quantiles (10/25/50/75/90%):")
for col in ['vcos_34', 'vcos_24', 'vcos_14', 'vcos_23']:
    print(f"  {col}: {np.nanpercentile(df[col], [10, 25, 50, 75, 90]).round(2)}")

In [ ]:
T_ = VOICE_SAME_COS
has = df.vcos_34.notna()
same34 = has & (df.vcos_34 > T_)
print(f"MCIS with clip III and IV voices: {has.sum()}")
print(f"  clip III and IV sound like the SAME speaker (violates A != B): {same34[has].mean() * 100:.1f}%")
valid = df[has & ~same34].copy()
valid['B_voice_II'] = valid.vcos_24 > T_
valid['B_voice_I'] = valid.vcos_14 > T_
valid['II_not_A'] = valid.vcos_23 <= T_          # inference-time rule: clip II is someone other than A
valid['I_not_A'] = valid.vcos_13 <= T_
valid['B_observable_v2'] = valid.B_listening_III_frontal | valid.B_voice_II | valid.B_voice_I


def block(d, name):
    print(f"\n== {name}: {len(d)} MCIS (A != B by voice) ==")
    print(f"  B spoke in clip II (voice)                 {d.B_voice_II.mean() * 100:5.1f}%")
    print(f"  B spoke in clip I  (voice)                 {d.B_voice_I.mean() * 100:5.1f}%")
    print(f"  B visible in III with usable face          {d.B_listening_III_frontal.mean() * 100:5.1f}%")
    print(f"  B observable (usable face OR voice turn)   {d.B_observable_v2.mean() * 100:5.1f}%")
    for k, name_ in [('2', 'II'), ('1', 'I')]:
        third = (d[f'vcos_{k}3'] <= T_) & (d[f'vcos_{k}4'] <= T_)
        print(f"  clip {name_:<2} speaker is a THIRD person (neither A nor B)  {third.mean() * 100:5.1f}%")
    for rule, truth in [('II_not_A', 'B_voice_II'), ('I_not_A', 'B_voice_I')]:
        fired = d[d[rule]]
        print(f"  rule '{rule}' fires {d[rule].mean() * 100:5.1f}% | precision {fired[truth].mean() * 100 if len(fired) else float('nan'):5.1f}%"
              f" | recall {d[d[truth]][rule].mean() * 100 if d[truth].any() else float('nan'):5.1f}%")


block(valid, "all")
block(valid[~valid.multi_party], "two-person scenes (<= 2 face identities)")
block(valid[valid.multi_party], "multi-person scenes (>= 3 face identities)")
print(f"\nmulti-person share: {df.multi_party.mean() * 100:.1f}% of analysed MCIS")
print("\nGATE: B observable >= 40-50%, rule precision >= 80%, and a small A==B (same voice) share.")
valid.to_csv(f"{OUT_DIR}/g6a_voice_valid.csv", index=False)

## Montages (send a few of these back for a visual check)
Rows = clips I–IV, columns = sampled frames. Box colours: **red** A (dominant in III), **green** B_true
(dominant in IV), **blue** the no-clip-IV prediction when it differs from B_true, **grey** other people.

In [ ]:
os.makedirs(f"{OUT_DIR}/g6a_montages", exist_ok=True)
for sid, (r, det) in DETAIL.items():
    if not det:
        continue
    cl = [r['clip1'], r['clip2'], r['clip3'], r['clip4']]
    person_at = {}
    for (k, f), p in zip(det['faces'], det['labels']):
        person_at.setdefault((cl[k], f['frame']), []).append((f['bbox'], int(p)))
    tiles_rows = []
    for k, c in enumerate(cl):
        tiles = []
        for fi, fr in KEEP_FRAMES.get(c, []):
            im = fr.copy()
            for (x1, y1, x2, y2), p in person_at.get((c, fi), []):
                col = ((0, 0, 255) if p == det['A'] else (0, 200, 0) if p == det['Bt'] else
                       (255, 120, 0) if p == det['Bp'] else (160, 160, 160))
                cv2.rectangle(im, (x1, y1), (x2, y2), col, 3)
                cv2.putText(im, str(p), (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
            tiles.append(cv2.resize(im, (256, 144)))
        while len(tiles) < 6:
            tiles.append(np.zeros((144, 256, 3), np.uint8))
        row_img = np.hstack(tiles[:6])
        cv2.putText(row_img, ['I', 'II', 'III (A speaks)', 'IV (B speaks)'][k], (5, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        tiles_rows.append(row_img)
    cv2.imwrite(f"{OUT_DIR}/g6a_montages/{sid}.jpg", np.vstack(tiles_rows))
print(len(os.listdir(f"{OUT_DIR}/g6a_montages")), "montages written")